In [ ]:
import os
import time
from typing import List, Optional
from pinecone import Pinecone, ServerlessSpec
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import UnstructuredMarkdownLoader, TextLoader

from Langchain_learning_2 import documents

from PineCone_test.PineCone_test import records
from sentence_transformers.util import normalize_embeddings


class RAG_Service:
    def __init__(
            self,
            index_name: str,
            api_key: Optional[str] = None,
            model_name: str = "Qwen/Qwen3-VL-Embedding-2B",
            dimension: int = 1536,  # Qwen2-1.5B 默认维度
            metric: str = "cosine",
            region: str = "us-east-1"
    ):
        self.index_name = index_name
        self.api_key = api_key or os.getenv("PINECONE_API_KEY")
        self.dimension = dimension
        self.metric = metric
        self.region = region

        # 1. 初始化 Pinecone 客户端
        self.pc = Pinecone(api_key=self.api_key)

        # 2. 初始化本地 Qwen Embedding 模型
        print(f"正在加载嵌入模型: {model_name}...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda'},
            encode_kwargs={'normalize_embeddings': True}
        )

        # 3. 初始化语义分块器
        self.text_splitter = SemanticChunker(self.embeddings, breakpoint_threshold_type="percentile")

        # 绑定索引操作对象
        if self.index_name in [idx.name for idx in self.pc.list_indexes()]:
            self.index = self.pc.Index(self.index_name)
        else:
            self.index = None

    def create_index(self):
        """创建索引，如果已存在则跳过"""
        if self.index_name not in [idx.name for idx in self.pc.list_indexes()]:
            print(f"正在创建索引: {self.index_name}...")
            self.pc.create_index(
                name=self.index_name,
                dimension=self.dimension,
                metric=self.metric,
                spec=ServerlessSpec(cloud="aws", region=self.region)
            )
            # 等待索引初始化完成
            while not self.pc.describe_index(self.index_name).status['ready']:
                time.sleep(1)

        self.index = self.pc.Index(self.index_name)
        print(f"索引 {self.index_name} 已就绪。")

    def add_Document(self, file_path: str, namespace: str = "default"):
        """载入 Markdown 文档，进行语义分块、向量化并上传"""
        if not self.index:
            raise ValueError("索引未初始化，请先调用 create_index()")

        print(f"正在处理文档: {file_path} ...")
        # 1. 加载 Markdown
        loader = UnstructuredMarkdownLoader(file_path)
        raw_docs = loader.load()

        # 2. 语义分块
        semantic_chunks = self.text_splitter.split_documents(raw_docs)

        # 3. 准备上传数据格式
        vectors_to_upsert = []
        for i, doc in enumerate(semantic_chunks):
            # 生成向量嵌入
            embedding = self.embeddings.embed_query(doc.page_content)

            # 这里的 ID 建议包含文件名以防重复
            doc_id = f"{os.path.basename(file_path)}_{i}"

            vectors_to_upsert.append({
                "id": doc_id,
                "values": embedding,
                "metadata": {
                    "text": doc.page_content,  # 必须存入原文以便搜索返回
                    "source": file_path
                }
            })

            # 分批上传，防止 payload 过大
            if len(vectors_to_upsert) >= 100:
                self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)
                vectors_to_upsert = []

        if vectors_to_upsert:
            self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)

        print(f"成功将 {len(semantic_chunks)} 个语义块上传至 Namespace: {namespace}")

    def Search_Vector(self, query: str, top_k: int = 3, namespace: str = "default"):
        """执行向量搜索并返回结果文本"""
        if not self.index:
            raise ValueError("索引未初始化")

        # 1. 向量化用户问题
        query_vector = self.embeddings.embed_query(query)

        # 2. 在 Pinecone 中检索
        search_results = self.index.query(
            vector=query_vector,
            top_k=top_k,
            namespace=namespace,
            include_metadata=True
        )

        # 3. 提取元数据中的文本内容
        context_list = []
        for res in search_results["matches"]:
            context_list.append({
                "text": res["metadata"]["text"],
                "score": res["score"],
                "source": res["metadata"]["source"]
            })

        return context_list

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

loader = TextLoader(
    "../lawApp_LangGraph/MarkDownFiles/中国法院2019年度案例：婚姻家庭与继承纠纷.md",
    encoding="utf-8"
)

headers_to_split_on = [("#", "Header_1")]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "；", "！", "？", " ", ""],  # 优先按段落切
    add_start_index=True
)

# 加载一次之后可以反复使用
# 稀疏向量
bm25 = BM25Encoder().load("../lawApp_LangGraph/RAG_service/bm25_law_params.json")

# 密集向量
model_name = "BAAI/bge-large-zh-v1.5"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs={'normalize_embeddings': True}
)

# 保存为本地数据
# global articles, metadata, facts_cleaned

# 插入Pinecone数据容器
Pinecone_records = []

documents = loader.load()

for doc in documents:
    # split_text 返回 List[Document]，每个 Document 对应一个一级标题下的内容块
    articles = md_splitter.split_text(doc.page_content)

    # 获取元数据和正文内容
    for i, article in enumerate(articles):
        print(f'正在处理第{i}文章\n')

        metadata = {}

        # 提取年份：假设文件名或路径包含 202X
        year_match = re.search(r"20\d{2}", article.page_content)
        if year_match:
            metadata["year"] = year_match.group(0) if year_match else "Unknown"

            # 提取裁判书字号：匹配如（2023）最高法民终...号
            case_num_pattern = r'裁判书字号[\s\\n]+((?:(?!裁判书字号)[\s\S])+?法院[\s\S]+?书)'
            case_num_match = re.search(case_num_pattern, article.page_content)
            metadata["case_number"] = case_num_match.group(1).strip() if case_num_match else "未识别"

            # 提取案由：通常在字号之后，或者是特定的段落
            case_cause_pattern = r"案由[:：]\s*([\u4e00-\u9fa5]+)"
            cause_match = re.search(case_cause_pattern, article.page_content)
            metadata["case_cause"] = cause_match.group(1) if cause_match else "通用"

            # 提取基本案情
            facts_pattern = r'【基本案情】\s*([\s\S]+?)(?=\n【|$)'
            facts_match = re.search(facts_pattern, article.page_content)

            # 获取捕获组内容（不含【基本案情】）
            raw_content = facts_match.group(1)
            # 去除空格、换行、制表符等所有空白字符，以及 # 符号
            facts_cleaned = re.sub(r'\n+', '\n', raw_content).strip()  # 去除所有空白（空格、换行等）
            facts_cleaned = facts_cleaned.replace('#', '')  # 去除所有 # 字符

            # ----- 立即对当前案例的案情文本进行切割和向量化 -----
            chunks = text_splitter.split_text(facts_cleaned)

            for chunk_id, chunk in enumerate(chunks):
                record_id = f"Docu{i}_chunk{chunk_id}"

                # 密集向量和稀疏向量
                dense_vector = embeddings.embed_query(chunk)

                # 返回 {"indices": [...], "values": [...]}
                sparse_vector = bm25.encode_documents(chunk)

                """添加内容: id 向量数据 元数据:{年份 判决书 案由 文档切片}"""
                record = {
                    "id": record_id,
                    "values": dense_vector,  # 稠密向量列表 [0.12, -0.34, ...]
                    "sparse_values": sparse_vector,  # 稀疏向量字典
                    "metadata": {
                        "chunk_index": chunk_id,
                        **metadata,
                        "chunk_text": chunk,
                    }
                }
                print(f'第{chunk_id}个record组装完毕')

                Pinecone_records.append(record)
        print("\n")
        print(f'第{i}文章处理完毕,已经添加至record中\n')





Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


正在处理第0文章
第0文章处理完毕,已经添加至record中

正在处理第1文章
第0个record组装完毕

第1个record组装完毕

第2个record组装完毕

第3个record组装完毕

第4个record组装完毕

第5个record组装完毕

第6个record组装完毕

第1文章处理完毕,已经添加至record中

正在处理第2文章
第0个record组装完毕

第2文章处理完毕,已经添加至record中

正在处理第3文章
第0个record组装完毕

第1个record组装完毕

第2个record组装完毕

第3个record组装完毕

第4个record组装完毕

第5个record组装完毕

第6个record组装完毕

第3文章处理完毕,已经添加至record中

正在处理第4文章
第0个record组装完毕

第1个record组装完毕

第2个record组装完毕

第3个record组装完毕

第4个record组装完毕

第5个record组装完毕

第4文章处理完毕,已经添加至record中

正在处理第5文章
第0个record组装完毕

第1个record组装完毕

第2个record组装完毕

第3个record组装完毕

第4个record组装完毕

第5个record组装完毕

第6个record组装完毕

第5文章处理完毕,已经添加至record中

正在处理第6文章
第0个record组装完毕

第1个record组装完毕

第2个record组装完毕

第3个record组装完毕

第4个record组装完毕

第5个record组装完毕

第6个record组装完毕

第7个record组装完毕

第8个record组装完毕

第6文章处理完毕,已经添加至record中

正在处理第7文章
第0个record组装完毕

第1个record组装完毕

第2个record组装完毕

第3个record组装完毕

第4个record组装完毕

第5个record组装完毕

第6个record组装完毕

第7个record组装完毕

第8个record组装完毕

第7文章处理完毕,已经添加至record中

正在处理第8文章
第0个record组装完毕

第1个record组装完毕

第2个record组装完毕



In [7]:
model_name = "BAAI/bge-large-zh-v1.5"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs={'normalize_embeddings': True}
)

query = "离婚"
query_vector = embeddings.embed_query(query)
print(query_vector)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[-0.019893163815140724, 0.043156594038009644, 0.023360176011919975, -0.049014050513505936, 0.012853655964136124, -0.018470440059900284, -0.04738694801926613, 0.0073791611939668655, -0.0010047397809103131, -0.04170503094792366, 0.046877019107341766, -0.000990700558759272, 0.024541204795241356, -0.023766446858644485, -0.005272245034575462, 0.030115416273474693, -0.004267795477062464, 0.047952957451343536, -0.03455560654401779, -0.02006983943283558, 0.010182240977883339, 0.013955502770841122, -0.030698200687766075, -0.013555224984884262, -0.00994021911174059, -0.0008978591649793088, -0.056955911219120026, 0.015700271353125572, 0.06780492514371872, -0.01849660649895668, -0.0016541085205972195, -0.0407901257276535, 0.008482111617922783, 0.05428583174943924, 0.0359143391251564, 0.01029936783015728, -0.005567943211644888, 0.016393296420574188, 0.011098885908722878, -0.03267882391810417, 0.03927825018763542, 0.013181460089981556, 0.0322430245578289, 0.033650755882263184, -0.015914078801870346,

In [5]:
from pinecone.grpc import PineconeGRPC as Pinecone
import os
from dotenv import load_dotenv

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("pinecone-test-lawapp")

# 分批上传，每批最多 100 条向量
batch_size = 50
total = len(Pinecone_records)

for i in range(0, total, batch_size):
    batch = Pinecone_records[i:i + batch_size]
    index.upsert(vectors=batch, namespace="Law_test_namespace")
    uploaded = min(i + batch_size, total)
    print(f"已上传 {uploaded}/{total} 条记录")

print(pc.describe_index("pinecone-test-lawapp"))

已上传 50/538 条记录
已上传 100/538 条记录
已上传 150/538 条记录
已上传 200/538 条记录
已上传 250/538 条记录
已上传 300/538 条记录
已上传 350/538 条记录
已上传 400/538 条记录
已上传 450/538 条记录
已上传 500/538 条记录
已上传 538/538 条记录
{'_response_info': {'raw_headers': {'access-control-allow-origin': '*',
                                    'access-control-expose-headers': '*',
                                    'alt-svc': 'h3=":443"; '
                                               'ma=2592000,h3-29=":443"; '
                                               'ma=2592000',
                                    'content-length': '414',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 25 Apr 2026 09:13:21 GMT',
                                    'server': 'Google Frontend',
                                    'vary': 'origin, '
                                            'access-control-request-method, '
                                            'access-control-request-header

In [ ]:
{'_response_info':
     {'raw_headers':
          {'access-control-allow-origin': '*',
           'access-control-expose-headers': '*',
           'alt-svc': 'h3=":443"; ma=2592000',
           'content-length': '414',
           'content-type': 'application/json',
           'date': 'Sat, 25 Apr 2026 03:10:31 GMT',
           'server': 'Google Frontend',
           'vary': 'origin, '
                   'access-control-request-method, '
                   'access-control-request-headers',
           'via': '1.1 google',
           'x-cloud-trace-context': 'ef29f06e5f74fc059f8e58bade84201a',
           'x-pinecone-api-version': '2025-10'}},
 'deletion_protection': 'disabled',
 'dimension': 1024,
 'host': 'pinecone-test-lawapp-ldu4w3a.svc.aped-4627-b74a.pinecone.io',
 'metric': 'dotproduct',
 'name': 'pinecone-test-lawapp',
 'spec': {
     'serverless':
         {'cloud': 'aws',
          'read_capacity': {
              'mode': 'OnDemand',
              'status': {'current_replicas': None,
                         'current_shards': None,
                         'state': 'Ready'}
          },
          'region': 'us-east-1'}},
 'status': {'ready': True, 'state': 'Ready'},
 'tags': None,
 'vector_type': 'dense'}


In [ ]:
# 安装依赖：pip install langchain-huggingface sentence_transformers

from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-large-zh-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# 单条文本嵌入
text = "这是一个测试文本。"
dense_vector = embeddings.embed_query(text)

# 多文本批量嵌入
texts = ["文本1", "文本2", "文本3"]
doc_vectors = embeddings.embed_documents(texts)
print(len(doc_vectors))

In [6]:
import re
import os
from pathlib import Path
from pinecone_text.sparse import BM25Encoder
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


def extract_facts_from_md(content: str) -> list[str]:
    """
    从 Markdown 文本中提取所有一级标题下【基本案情】的内容
    返回清洗后的文本列表
    """
    headers_to_split_on = [("#", "Header_1")]
    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    articles = md_splitter.split_text(content)

    facts_list = []
    for article in articles:
        # 在每个案例中搜索【基本案情】
        facts_pattern = r'【基本案情】\s*([\s\S]+?)(?=\n【|$)'
        match = re.search(facts_pattern, article.page_content)
        if match:
            raw = match.group(1)
            cleaned = re.sub(r'\n+', ' ', raw).strip()  # 合并换行
            cleaned = cleaned.replace('#', '')
            facts_list.append(cleaned)

    return facts_list


# 1. 收集所有 .md 文件路径
md_dir = Path("../lawApp_LangGraph/MarkDownFiles")
md_files = list(md_dir.glob("*.md"))
print(f"找到 {len(md_files)} 个Markdown文件")


找到 11 个Markdown文件


In [5]:

# 2. 提取所有“基本案情”文本
all_facts = []
for file_path in md_files:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    facts = extract_facts_from_md(content)

    if facts:
        all_facts.extend(facts)
    else:
        print(f"警告：{file_path.name} 未找到【基本案情】段落")

print(f"成功提取 {len(all_facts)} 段基本案情")


成功提取 689 段基本案情


In [7]:

# 3. 一次性训练 BM25
bm25 = BM25Encoder()
bm25.fit(all_facts)  # 拟合IDF等参数

# 4. 保存训练好的参数（指定路径）
save_path = "lawApp_LangGraph/pdf_output/bm25_law_params.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
bm25.dump(save_path)

print(f"BM25 参数已保存至：{save_path}")

  0%|          | 0/689 [00:00<?, ?it/s]

BM25 参数已保存至：lawApp_LangGraph/pdf_output/bm25_law_params.json


In [6]:
from pinecone import Pinecone
import os
from dotenv import load_dotenv

load_dotenv()
Pinecone_api_key = os.getenv("PINECONE_API_KEY")
PC = Pinecone(api_key=Pinecone_api_key)
PC.describe_index("my-rag-index")

{
    "name": "my-rag-index",
    "metric": "cosine",
    "host": "my-rag-index-ldu4w3a.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2

In [ ]:
from pinecone import Pinecone, SparseValues, Vector

pc = Pinecone(api_key="YOUR_API_KEY")

# To get the unique host for an index,
# see https://docs.pinecone.io/guides/manage-data/target-an-index


"""

上传稀疏向量数据
相比密集向量
新增indices chunk_text
也就是说
上传稀疏向量同时需要添加文本和向量

"""
index = pc.Index(host="INDEX_HOST")

index.upsert(
    namespace="example-namespace",
    vectors=[
        {
            "id": "vec1",
            "sparse_values": {
                "values": [1.7958984, 0.41577148, 2.828125, 2.8027344, 2.8691406, 1.6533203, 5.3671875, 1.3046875,
                           0.49780273, 0.5722656, 2.71875, 3.0820312, 2.5019531, 4.4414062, 3.3554688],
                "indices": [822745112, 1009084850, 1221765879, 1408993854, 1504846510, 1596856843, 1640781426,
                            1656251611, 1807131503, 2543655733, 2902766088, 2909307736, 3246437992, 3517203014,
                            3590924191]
            },
            "metadata": {
                "chunk_text": "AAPL reported a year-over-year revenue increase, expecting stronger Q3 demand for its flagship phones.",
                "category": "technology",
                "quarter": "Q3"
            }
        },
        {
            "id": "vec2",
            "sparse_values": {
                "values": [0.4362793, 3.3457031, 2.7714844, 3.0273438, 3.3164062, 5.6015625, 2.4863281, 0.38134766,
                           1.25, 2.9609375, 0.34179688, 1.4306641, 0.34375, 3.3613281, 1.4404297, 2.2558594, 2.2597656,
                           4.8710938, 0.5605469],
                "indices": [131900689, 592326839, 710158994, 838729363, 1304885087, 1640781426, 1690623792, 1807131503,
                            2066971792, 2428553208, 2548600401, 2577534050, 3162218338, 3319279674, 3343062801,
                            3476647774, 3485013322, 3517203014, 4283091697]
            },
            "metadata": {
                "chunk_text": "Analysts suggest that AAPL'\''s upcoming Q4 product launch event might solidify its position in the premium smartphone market.",
                "category": "technology",
                "quarter": "Q4"
            }
        },
        {
            "id": "vec3",
            "sparse_values": {
                "values": [2.6875, 4.2929688, 3.609375, 3.0722656, 2.1152344, 5.78125, 3.7460938, 3.7363281, 1.2695312,
                           3.4824219, 0.7207031, 0.0826416, 4.671875, 3.7011719, 2.796875, 0.61621094],
                "indices": [8661920, 350356213, 391213188, 554637446, 1024951234, 1640781426, 1780689102, 1799010313,
                            2194093370, 2632344667, 2641553256, 2779594451, 3517203014, 3543799498, 3837503950,
                            4283091697]
            },
            "metadata": {
                "chunk_text": "AAPL'\''s strategic Q3 partnerships with semiconductor suppliers could mitigate component risks and stabilize iPhone production",
                "category": "technology",
                "quarter": "Q3"
            }
        },
        {
            "id": "vec4",
            "sparse_values": {
                "values": [0.73046875, 0.46972656, 2.84375, 5.2265625, 3.3242188, 1.9863281, 0.9511719, 0.5019531,
                           4.4257812, 3.4277344, 0.41308594, 4.3242188, 2.4179688, 3.1757812, 1.0224609, 2.0585938,
                           2.5859375],
                "indices": [131900689, 152217691, 441495248, 1640781426, 1851149807, 2263326288, 2502307765, 2641553256,
                            2684780967, 2966813704, 3162218338, 3283104238, 3488055477, 3530642888, 3888762515,
                            4152503047, 4177290673]
            },
            "metadata": {
                "chunk_text": "AAPL may consider healthcare integrations in Q4 to compete with tech rivals entering the consumer wellness space.",
                "category": "technology",
                "quarter": "Q4"
            }
        }
    ]
)

In [5]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_community.retrievers import (
    PineconeHybridSearchRetriever,
)
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

load_dotenv()

# 1. 初始化环境变量
index_name = "pinecone-test-lawapp"

# 2. 初始化 Pinecone 客户端
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# 3. 创建索引 (如果不存在)
if index_name not in pc.list_indexes().names():
    # pc.create_index(
    #     name=index_name,
    #     dimension=1024,  # BGE-M3 维度
    #     metric="dotproduct",  # 混合检索必须用 dotproduct
    #     spec=ServerlessSpec(cloud="aws", region="us-east-1")
    # )
    print("索引index不存在")

# 4. 初始化稠密向量模型 (BGE-M3)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 5. 初始化稀疏向量编码器 (BM25)

bm25_encoder = BM25Encoder().default()
bm25_encoder.load("bm25_law_params.json")

# 6. 构建 LangChain 检索器
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=pc.Index(index_name),
    top_k=10,
    alpha=0.4  # 1偏向语义，0偏向关键词匹配
)

# 7. 执行查询
query = "离婚子女抚养权"
docs = retriever.invoke(query)

for doc in docs:
    print(f"找到条文：{doc.page_content[:100]}...")
    print("查找的原文:", doc)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [1]:
import re

text = """
1. 裁判书字号  \n北京市第三中级人民法院(2017)京03民终第8341号民事判决书
"""

pattern = r'\d+\.\s+裁判书字号\s*\n\s*(.+)'
matches = re.findall(pattern, text)

for m in matches:
    print(m)

北京市第三中级人民法院(2017)京03民终第8341号民事判决书


In [2]:
from transformers import EncoderDecoderCache, Trainer
from peft import PeftModel

print("导入成功！")

导入成功！


In [ ]:
pip
install
weasyprint
markdown
pygments

In [ ]:
from datetime import datetime
import markdown
import os
from weasyprint import HTML


def markdown_to_pdf(markdown_text_filePath: str, filename: str = None) -> str:
    # 1. 读取 Markdown 文件
    with open(markdown_text_filePath, "r", encoding="utf-8") as f:
        markdown_text = f.read()

    # 2. Markdown → HTML（保留表格、代码高亮）
    html_body = markdown.markdown(
        markdown_text,
        extensions=['extra', 'codehilite']
    )

    # 3. 嵌入 CSS，确保中文排版
    styled_html = f"""
    <!DOCTYPE html>
    <html lang="zh-CN">
    <head>
        <meta charset="UTF-8">
        <style>
            @page {{
                size: A4;
                margin: 2cm;
            }}
            body {{
                font-family: 'SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei', sans-serif;
                font-size: 14px;
                line-height: 1.6;
            }}
            h1 {{ color: #333; }}
            pre {{ background: #f4f4f4; padding: 10px; border-radius: 5px; }}
            code {{ font-family: 'Courier New', monospace; }}
        </style>
    </head>
    <body>{html_body}</body>
    </html>
    """

    # 4. 生成文件名
    if not filename:
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"

    # 5. 保存 PDF
    output_dir = "../lawApp_LangGraph/pdf_output"
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, filename)

    HTML(string=styled_html).write_pdf(file_path)

    return f"PDF 已成功生成，文件路径：{file_path}"


def markdown_to_html(markdown_text: str) -> str:
    return markdown.markdown(markdown_text, extensions=['extra', 'codehilite'])


# 测试
print(markdown_to_pdf("../lawApp_LangGraph/MarkDownFiles/中国法院2014年度案例_婚姻家庭与继承纠纷.md",
                      "中国法院2014年度案例_婚姻家庭与继承纠纷"))

## 测试语义向量检索
* 引入BGE向量化
* 实现问题的向量化
* 实现密集向量查询

## 测试关键字检索
* 引入BM25
* 稀疏向量化
* 实现稀疏矩阵查询


In [18]:
import os
from typing import Any
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
import time
from pinecone import Pinecone, ServerlessSpec
from pinecone.grpc import PineconeGRPC as Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index = pc.Index(host="https://pinecone-test-lawapp-ldu4w3a.svc.aped-4627-b74a.pinecone.io")

# 稀疏向量
bm25_encoder = BM25Encoder().default()
bm25_encoder.load("bm25_law_params.json")

# 密集向量
model_name = "BAAI/bge-large-zh-v1.5"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs={'normalize_embeddings': True}
)

# 不要混淆 index和namespace

namespace = "Law_test_namespace"
query = "离婚子女抚养权"

dense_query_vec = embeddings.embed_query(query)

# dense_result = index.query(
#     vector=dense_query_vec,
#     top_k=5,
#     namespace=namespace,
#     include_metadata=True
# ).matches
#
# print(f"密集检索的结果:{dense_result},其类型为:{type(dense_result)}")

sparse_query_vec = bm25.encode_queries(query)
# sparse_result = index.query(
#     sparse_vector=sparse_query_vec,
#     top_k=5,
#     namespace=namespace,
#     include_metadata=True
# ).matches
#
# print(f"稀疏检索的结果:{sparse_result},其类型为:{type(sparse_result)}")

result = index.query(
    vector=dense_query_vec,           # 必须提供稠密向量
    sparse_vector=sparse_query_vec,   # 可选的稀疏向量，用于提升召回
    top_k=5,
    namespace=namespace,
    include_metadata=True
)

print("融合的检索结果:",result)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


融合的检索结果: QueryResponse(matches=[{'id': 'Docu30_chunk6',
 'metadata': {'case_cause': '抚养费纠纷',
              'case_number': '福建省厦门市中级人民法院(2017)闽02民终第2193号民事判决书',
              'chunk_index': 6.0,
              'chunk_text': '目前，主张子女抚养费的主体有两类情况，一类是子女作为原告起诉，另一类是离异双方中与子女共同生活的一方即父母其中一方作为原告起诉。两类主体在司法实践中都普遍存在，第一类主体的原告诉讼主体资格毋庸置疑，而第二类主体则有人主张不具有原告诉讼主体资格。本文的观点是，两类主体还是要区别对待，虽然都是主张抚养费，但由谁主张，法律依据有所不同。作为子女向父母一方主张抚养费，直接根据民法、婚姻法的相关规定，而在离异双方就子女抚养费进行书面协议约定的情形下，作为离异双方中一方有权基于协议作为原告向另一方主张子女抚养费。两者不能混同，特别是本案的情形，是否支持利息，既已适合合同法予以判决，则主体也应符合合同相对性原则，离婚协议虽然为未成年子女设定的权益，但其约束力仍应在离婚双方。因此，在原告主体资格方面区分是必要的，否则会形成逻辑上的矛盾。子女作为原告起诉时，根据《中华人民共和国婚姻法》第三十七条第二款的规定，可以按父母离婚协议中约定的抚养费标准提出抚养费主张，但其并不是离婚协议的双方当事人，不能直接按协议约定主张相关的迟延履行利息',
              'year': '2017'},
 'score': 0.6828470230102539,
 'sparse_values': None,
 'values': []}, {'id': 'Docu33_chunk8',
 'metadata': {'case_cause': '变更抚养关系纠纷',
              'case_number': '福建省泉州市中级人民法院(2017)闽05民终第604号民事判决书',
              'chunk_index': 8.0,
              'chunk_text': '2. 如何证明一方不尽抚养义务？ 

In [ ]:

融合的检索结果: QueryResponse(
    matches=[
        {'id': 'Docu30_chunk6',
 'metadata': {'case_cause': '抚养费纠纷',
              'case_number': '福建省厦门市中级人民法院(2017)闽02民终第2193号民事判决书',
              'chunk_index': 6.0,
              'chunk_text': '目前，主张子女抚养费的主体有两类情况，一类是子女作为原告起诉，另一类是离异双方中与子女共同生活的一方即父母其中一方作为原告起诉。两类主体在司法实践中都普遍存在，第一类主体的原告诉讼主体资格毋庸置疑，而第二类主体则有人主张不具有原告诉讼主体资格。本文的观点是，两类主体还是要区别对待，虽然都是主张抚养费，但由谁主张，法律依据有所不同。作为子女向父母一方主张抚养费，直接根据民法、婚姻法的相关规定，而在离异双方就子女抚养费进行书面协议约定的情形下，作为离异双方中一方有权基于协议作为原告向另一方主张子女抚养费。两者不能混同，特别是本案的情形，是否支持利息，既已适合合同法予以判决，则主体也应符合合同相对性原则，离婚协议虽然为未成年子女设定的权益，但其约束力仍应在离婚双方。因此，在原告主体资格方面区分是必要的，否则会形成逻辑上的矛盾。子女作为原告起诉时，根据《中华人民共和国婚姻法》第三十七条第二款的规定，可以按父母离婚协议中约定的抚养费标准提出抚养费主张，但其并不是离婚协议的双方当事人，不能直接按协议约定主张相关的迟延履行利息',
              'year': '2017'},
 'score': 0.6828470230102539,
 'sparse_values': None,
 'values': []}, {'id': 'Docu33_chunk8',
 'metadata': {'case_cause': '变更抚养关系纠纷',
              'case_number': '福建省泉州市中级人民法院(2017)闽05民终第604号民事判决书',
              'chunk_index': 8.0,
              'chunk_text': '2. 如何证明一方不尽抚养义务？  \n'
                            '《最高人民法院关于人民法院审理离婚案件处理子女抚养问题的若干具体意见》第十六条第（二）项规定：与子女共同生活的一方不尽抚养义务或有虐待子女行为，或其与子女共同生活对子女身心健康却有不利影响的，一方要求变更抚养关系，应予支持。但证明一方不尽义务需要哪些证据来证实，也是当事人面临的实际问题。编者以为可以从三个方面进行举证：(1)居住状况改变：例如，双方离婚后，抚养子女的一方已经没有与子女共同生活，将子女交托（外）祖父母进行抚养，或者是已再婚无法同子女共同生活。（2）经济能力变化：例如，双方离婚时，约定或者判定抚养子女的一方，在时间推移中经济能力已经不足以抚养子女，而另外一方经济能力较好，足以抚养子女。（3）有利于被抚养子女的健康成长：在双方经济能力、居住情况都相当的情况下，被抚养子女的心理因素成为承办法官应当考虑的重要因素。相对上述两条，这条的举证相对更为困难，如若是年满十周岁以上的未成年人可以询问其意见，但多数离婚案件或者变更抚养关系的案件，子女都不见得达到自主意识明确的年纪，其行为能力和判断能力都相对较弱。这就要求当事人从有益于孩子身心发展的方面进行举证。',
              'year': '2017'},
 'score': 0.679692268371582,
 'sparse_values': None,
 'values': []}, {'id': 'Docu33_chunk2',
 'metadata': {'case_cause': '变更抚养关系纠纷',
              'case_number': '福建省泉州市中级人民法院(2017)闽05民终第604号民事判决书',
              'chunk_index': 2.0,
              'chunk_text': '【案件焦点】  \n'
                            '原告是否有必要请求变更婚生子的抚养关系。  \n'
                            ' 【法院裁判要旨】  \n'
                            '福建省泉州市安溪县人民法院经审理认为:原告程某娥与被告苏某练于2014年12月30日自愿协议离婚并达成《离婚协议书》，系双方的真实意思表示，应受法律保护。《离婚协议书》约定:婚生子苏某某由苏某练负责监护抚养，随同男方生活。根据法律规定，父母与子女间的关系，不因父母离婚而消除。离婚后，子女无论由父或母直接抚养，仍是父母双方的子女。离婚后，父母对子女仍有抚养和教育的权利和义务。本案中，原告程某娥提供的证据虽可以证明苏某练负债的部分事实，以及程某娥经营晋江市永和镇某某饰品店和在民生银行有理财资金的事实，但尚不足以充分证明现抚养人即被告苏某练存在法律规定的应当变更子女抚养关系的情形，故对原告程某娥要求变更子女抚养关系的请求，本院不予支持。  \n'
                            '福建省泉州市安溪县人民法院依照《中华人民共和国婚姻法》第三十六条，《中华人民共和国民事诉讼法》第六十四条第一款、第一百  \n'
                            '四十四条，《最高人民法院关于人民法院审理离婚案件处理子女抚养问题的若干具体意见》第十六条，《最高人民法院关于民事诉讼证据的若干规定》第二条之规定，作出如下判决：',
              'year': '2017'},
 'score': 0.6781949996948242,
 'sparse_values': None,
 'values': []}, {'id': 'Docu25_chunk6',
 'metadata': {'case_cause': '同居关系子女抚养纠纷',
              'case_number': '福建省福清市人民法院(2017)闽0181民初第4447号民事判决书',
              'chunk_index': 6.0,
              'chunk_text': '根据《中华人民共和国婚姻法 '
                            '》第二十一条之规定，父母对子女有抚养教育的义务；子女对父母有赡养扶助的义务。父母不履行抚养义务时，未成年的或不能独立生活的子女，有要求父母付给抚养费的权利。子女不履行赡养义务时，无劳动能力的或生活困难的父母，有要求子女付给赡养费的权利。《最高人民法院关于人民法院审理离婚案件处理子女抚养问题的若干具体意见》第十一条规定，抚育费的给付期限，一般至子女十八周岁为止。十六周岁以上不满十八周岁，以其劳动收入为主要生活来源，并能维持当地一般生活水平的，父母可停止给付抚育费。《最高人民法院关于适用〈中华人民共和国婚姻法〉若干问题的解释（一）》第二十一条规定的婚姻法第二十一条所称“抚养费”，包括子女生活费、教育费、医疗费等费用。抚养费的法律属性和特点有三：(1)抚养义务的无条件性。父母对未成年子女的抚养义务是无条件的，子女一旦出生，无论父母经济条件及负担能力如何，也不论是否愿意，均必须依法承担抚养义务。即使降低自己的生活水平、牺牲自己的事业发展和生活享受，也必须首先保障子女的生存和生活。(2)抚养内容的复合性',
              'year': '2017'},
 'score': 0.673405647277832,
 'sparse_values': None,
 'values': []}, {'id': 'Docu28_chunk1',
 'metadata': {'case_cause': '抚养费纠纷',
              'case_number': '四川省凉山州越西县人民法院(2017)川3434民初第1017号民事判决书',
              'chunk_index': 1.0,
              'chunk_text': '四川省越西县人民法院经审理认为:本案系抚养费纠纷。夫妻双方离婚后，父母应当对子女的抚养和教育承担自己的义务，在父母不履行抚养义务时，未成年的或不能独立生活的子女，有要求父母付给抚养费的权利。《中华人民共和国婚姻法》第二十一条第一款、第二款规定：“父母对子女有抚养教育的义务；……父母不履行抚养义务时，未成年的或者不能独立生活的子女，有要求父母付给抚养费的权利。”因此，子女若想通过法律途径要求父母抚养，必须具备两个要件：一是尚未成年，即年龄未满18周岁；二是虽然已经成年，但是不能独立生活。对何谓不能独立生活，《最高人民法院关于适用〈中华人民共和国婚姻法〉若干问题的解释（一）》第二十条做了进一步的明确规定：“婚姻法第二十一条规定的‘不能独立生活的子女’，是指尚在校接受高中及其以下学历教育，或者丧失或未完全丧失劳动能力等非因主观原因而无法维持正常生活的成年子女',
              'year': '2017'},
 'score': 0.6711206436157227,
 'sparse_values': None,
 'values': []}], namespace='Law_test_namespace', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Mon, 27 Apr 2026 03:31:17 GMT', 'x-pinecone-max-indexed-lsn': '12', 'x-pinecone-request-latency-ms': '11', 'x-envoy-upstream-service-time': '11', 'x-pinecone-response-duration-ms': '12', 'server': 'envoy'}})


In [16]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

load_dotenv()

# ---- 基本配置 ----
index_name = "pinecone-test-lawapp"
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(index_name)

# 密集编码器
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",  # BGE-M3 支持更长的上下文，比 bge-large-zh-v1.5 更适合混合检索
    encode_kwargs={'normalize_embeddings': True}
)

# 稀疏编码器（加载预训练参数）
bm25_encoder = BM25Encoder().default()
bm25_encoder.load("bm25_law_params.json")

# ---- 创建混合检索器 ----
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
    top_k=5,
    alpha=0.4  # 1=纯语义，0=纯关键词，0.4 兼顾两者
)

# ---- 执行查询 ----
query = "离婚子女抚养权"
results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)
    print("---")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:
import os
from datetime import datetime
import markdown
import pdfkit
from langchain_community.utilities import SerpAPIWrapper
from dotenv import load_dotenv

os.add_dll_directory(r'F:\GTK3-Runtime Win64\bin')
from weasyprint import HTML

load_dotenv()

def get_google_search(query: str):
    """使用谷歌搜索API在线搜索信息,适用于回答时事、确认事实或寻找特定网址"""

    serpapi_api_key=os.getenv("SERPAPI_API_KEY")
    print(serpapi_api_key)
    # 环境变量中已设置 SERPAPI_API_KEY
    search = SerpAPIWrapper(serpapi_api_key=serpapi_api_key)

    # 获取结构化结果
    results = search.results(query)

    # 打印完整返回（用于调试）
    print("SerpAPI 原始响应：", results.keys())
    if "error" in results:
        print("查到错误：", results["error"])
    if "organic_results" not in results:
        print("注意：响应中没有 'organic_results' 字段，可能是搜索被拦截或查询问题。")
        print("全部结果键：", list(results.keys()))

    # 格式化输出，方便 LLM 阅读
    formatted_results = []
    for res in results.get("organic_results", [])[:5]:
        formatted_results.append(
            f"标题: {res.get('title')}\n摘要: {res.get('snippet')}\n链接: {res.get('link')}\n---"
        )
    if not formatted_results:
        return "未找到相关搜索结果。"

    return "\n".join(formatted_results)


def markdown_to_html(markdown_text: str) -> str:
    """将Markdown文本转换为HTML字符串，并启用表格等扩展功能"""
    # 'extra' 扩展支持了表格、围栏代码块等常用Markdown语法

    return markdown.markdown(markdown_text, extensions=['extra', 'codehilite'])


# MarkDown文件转为pdf
def markdown_to_pdf(markdown_text: str, filename: str = None) -> str:
    """
    MarkDown文件转为pdf,当用户指定pdf文件输出时使用

    参数:
    markdown文本 , 文件名

    返回:
    返回文件存放路径

    :param markdown_text:
    :param filename:
    :return:
    """
    # 生成文件名
    if not filename:
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    else:
        filename = f"{filename}.pdf"

    # 将 Markdown 转换为 HTML
    html_content = markdown_to_html(markdown_text)

    # --- 添加 CSS 样式，解决中文乱码和排版问题 ---
    styled_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <style>
            body {{ font-family: 'SimHei', 'Microsoft YaHei', sans-serif; margin: 1cm; }}
            h1 {{ color: #333; }}
            code {{ font-family: monospace; background-color: #f4f4f4; }}
            pre {{ background-color: #f4f4f4; padding: 10px; border-radius: 5px; }}
        </style>
    </head>
    <body>
        {html_content}
    </body>
    </html>
    """

    # 配置 PDF 选项
    options = {
        'page-size': 'A4',  # 页面大小
        'margin-top': '0.75in',  # 上边距
        'margin-right': '0.75in',  # 右边距
        'margin-bottom': '0.75in',  # 下边距
        'margin-left': '0.75in',  # 左边距
        'encoding': "UTF-8",  # 编码
        'no-outline': None  # 无轮廓
    }

    # 确保输出目录存在
    output_dir = "./pdf_outputs"
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, filename)


    # 使用 weasyprint 生成 PDF
    HTML(string=styled_html).write_pdf(file_path)

    return f"PDF 已成功生成，文件路径：{file_path}"



In [5]:
results = get_google_search("什么是大模型(llm)")

print("网络搜索结果为: ",results)

b7b5a7243a22d933a5ddc2b82d88c31ca29b7f108079324bfa888be8b65fec49
SerpAPI 原始响应： dict_keys(['search_metadata', 'search_parameters', 'search_information', 'inline_images', 'ai_overview', 'answer_box', 'organic_results', 'related_searches', 'pagination', 'serpapi_pagination'])
网络搜索结果为:  标题: 什么是LLM？— 大型语言模型简介
摘要: 大型语言模型，也称为LLM，是基于大量数据进行预训练的超大型深度学习模型。底层转换器是一组神经网络，这些神经网络由具有自注意力功能的编码器和解码器组成。
链接: https://aws.amazon.com/cn/what-is/large-language-model/
---
标题: 什么是LLM大语言模型？Large Language Model，从量变到 ...
摘要: 大语言模型（英文：Large Language Model，缩写LLM），也称大型语言模型，是一种人工智能模型，旨在理解和生成人类语言。它们在大量的文本数据上进行训练，可以 ...
链接: https://zhuanlan.zhihu.com/p/622518771
---
标题: LLM：什么是大语言模型？ | Machine Learning
摘要: 一种较新的技术，即大语言模型(LLM)，可预测一个token 或一系列token，有时甚至可以预测相当于多个段落的token。请注意，词元可以是字词、子字词（字词的子集），甚至是单个 ...
链接: https://developers.google.com/machine-learning/crash-course/llm/transformers?hl=zh-cn
---
标题: 大型语言模型- 维基百科，自由的百科全书
摘要: 大型语言模型（英语：large language model，LLM），也称大语言模型，简称大模型，是一种基于人工神经网络的语言模型。其名称中的“大型”指模型具有庞大的参数量（通常在数十亿 ...
链接

In [3]:
with open('E:\LangChain\lawApp_LangGraph\MarkDownFiles\中国法院2021年度案例：婚姻家庭与继承纠纷.md',encoding='utf-8') as f:
    content = f.read()
    print(content[:500])

markdown_to_pdf(content,"中国法院2021年度案例")

## 一、婚约财产纠纷

# 彩礼返还案件中“婚前给付并导致给付人 生活困难”的合理认定 ——张某甲诉张某乙婚约财产案

#### 【案件基本信息】

1. 裁判书字号

山东省淄博市中级人民法院（2019）鲁03民终750号民事判决书

2. 案由：婚约财产纠纷

3. 当事人

原告（被上诉人）：张某甲

被告（上诉人）：张某乙

##### 【基本案情】

2016年12月6日，张某甲与张某乙依据当地传统习俗举行了订婚仪式，张某甲按照习俗给付张某乙彩礼61800元。双方于2017年2月27日登记结婚。张某乙于2017年5月中旬从张某甲家离开。2018年6月22日，经法院调解张某甲与张某乙离婚。张某甲的父母均为残疾人，母亲患有乳腺癌，家庭生活困难。张某甲起诉要求依法判令张某乙返还彩礼。张某乙辩称双方已在一起共同生活，涉案彩礼大部分用于置办结婚用品及共同生活的开支费用，即使尚有剩余也不应予以返还，本案不属于法律规定的应当予以返还彩礼的情形。

##### 【案件焦点】

张某甲要求张某乙返还彩礼有无事实和法律依据。

##### 【法院裁判要旨】

山东省桓台县人民法院经审理认为：双


'PDF 已成功生成，文件路径：./pdf_outputs\\中国法院2021年度案例.pdf'

In [ ]:
{
    'chunk_index': 0,
    'id': 'doc_chunk_0',
    'metadata':
        {'case_cause': '人身安全保护令',
         'case_number': '云南省昭通市鲁甸县人民法院(2017)云0621民保令第2号民事裁定书',
         'chunk_text': '申请人耿某某与被申请人曹某某均系教师，二人于2011年确立恋爱关系，于2013年10月14日办理结婚登记手续，于2014年5月18日生育一子曹某涵。婚后夫妻因家务琐事发生争吵，2015年9月、2016年5月，申请人耿某某先后两次被被申请人曹某某打伤。2017年9月23日，申请',
         'year': '2017'},
    'sparse_values': {
        'indices': [1826951998, 3302586229, 4280047764, 2906185329, 3557921014, 1728204391, 550526999, 2068246531,
                    2306698243, 3251671578, 287991880, 1449754254, 2694352034],
        'values': [0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659,
                   0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659,
                   0.6625503854497659, 0.6625503854497659, 0.6625503854497659]
    },
    'values': [-0.03657614812254906, 0.04955880716443062, 0.04295945540070534, 0.016933824867010117,
               0.019821351394057274,
               0.0 ** 14890265465, 0.016650019213557243,
               0.015882203355431557, -0.009750094264745712, 0.005802732892334461, 0.002266672905534506,
               -0.011243334040045738, -0.011120816692709923, -0.016657091677188873, 0.001065897406078875,
               0.012090726755559444, -0.02417769469320774, 0.008060940541327, 0.03416145592927933,
               -0.004575522616505623,
               -0.02823040261864662, -0.02307465858757496]
}